# Preparación del dataset para aplicar ML
**Propósito**: Cruzar los datos obtenidos de Idealista (Murcia) y los datos del CREM y darles un formato adecuado para aplicar ciencia de datos, y entrenar modelos.

In [1]:
import sys
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils

Lo que hacemos aquí será cargar los datos y filtrar y limpiar aquellos que puedan suponer un problema para el siguiente estudio.

- Quitamos los anuncios con más del 6 habitaciones, son muy pocos anuncios, suponen menos del 3% pueden suponer un problema, representando outliers. Hay incluso anuncios de inmuebles de más de 10 habitaciones; haciendo una pequeña cata se puede observar que o bien no tiene sentido por los metros cuadrados, por mucho que así lo indique el anuncio, o bien son propiedades con características "extrañas" como pueden ser, por ejemplo, casas dobles.
- Se mantienen los inmuebles con 0 habitaciones, se trata de lofts o estudios, son pocos pero decido mantenerlos.
- Nos quedamos con aquellos registros con un número de baños entre 1 y 4, la suma de aquellos anuncios con 0 o 5 o más baños, la suma de todos estos registros suponen cerca del 2.5%, pueden representar outliers.
- Filtramos el campo *numero_plantas*, quitamos aquellos que tienen 4 plantas o más, en su conjunto representan menos de 1% y además, los casos con más de 4 plantas son muy reducidos, y revisandolos, hay varios con ese campo mal rellenado por los vendedores.
- Redefinimos la columna de tipo de inmueble para pasar de todas las catégorias actuales a diferenciar si es un piso o no lo es.
- Redefinimos la columna de tipo de vendedor para que sea 1 si es un vendedor particular, o si es agencia.
- Se descarta el campo de *metros_cuadrados_utiles*, era de esperar que hubiera registros con este campo vacío, la idea inicial era rellenar este campo con el productro de *metros_cuadrados_construidos* por la media entre los metros construidos y los útiles para aquellos que sí tienen ambos campos (*0.835137*), pero la cantidad de inmuebles con este campo vacío es más la mitad del total, rellenarlo de esta forma puede crear un perjuicio, hacer esto, y más con este tamaño, supone que una reducción enorme de la variabilidad del campo y puede suponer que el modelo final sea sentible a los outlier.
- Se descarta el campo *planta_piso* no aporta mucho, por como hemos construido el prompt si no se dice en el anuncio es null y hay muchos, además, cuando está indicado no aporta demasiado, hay otros dos campos que a mi parecer pueden ser más decisivos en el precio de un inmueble, si es casa, si es ático, o si es un bajo.
- Descartamos la columna *prev_price* esta tiene el valor previo (en caso de haberse modificado), para el dataset restante esta columna es nula.
- Descartamos la columna *price_per_m2* al estar directamente relacionada con el campo de precio que queremos predecir.


In [2]:
#Inicializar sesión de Spark
spark = tfm_utils.get_spark_session(app_name="ML_create_dataset")

df_crem=tfm_utils.read_from_delta(spark, "silver.crem.crem_silver_full_data")
        

df_idealista=(tfm_utils.read_from_delta(spark, "silver.idealista.detalle_propiedades")
                .filter(F.col("provincia")=="murcia")
                #Nos quedamos con los procesados por IA
                .filter(F.col("_processed_ai")=="1")
                #Quitamos los solares
                .filter(F.col("es_solar")==0)
                #Quitamos aquellas viviendas con mas de 6 habitaciones, son un 3% y pueden representar outliers
                .filter(F.col("habitaciones")<=6)
                #Filtramos por numero de baños
                .filter(F.col("numero_banos").between(1,4))
                #Filtramos los anuncios por número de plantas 
                .filter(F.col("numero_plantas")<4)#-------10.959
                #Construimos una columna para saber si un inmueble es un piso (1) o una casa (0)
                .withColumn("es_piso", F.when(F.col("tipo_inmueble").isin(["apartamento","piso","estudio","atico","ático"]), F.lit(1))
                                            .otherwise(F.lit(0))
                           )
                #Definimos la columna es_vendedor_particular con 1 si lo es y 0 si es agencia
                .withColumn("es_vendedor_particular", F.when(F.col("tipo_vendedor").isin(["agencia"]), F.lit(0))
                                            .otherwise(F.lit(1))
                           )
                #Borramos columnas innecesarias o que no queremos
                .drop("provincia", "_processed_ai", "es_solar", "metros_cuadrados_utiles" ,"tipo_inmueble" 
                      ,'tipo_vendedor' ,'url' ,'planta_piso' ,'prev_price', 'price_per_m2')                
        )

## Cruzamos los datos de Idealista y el CREM

Cuando se hace el cruce se pierden 450, dentro de la parte de Idealista tenemos el municipio de *La Manga Del Mar Menor* que no tiene correspondencia en la parte del CREM, parte donde tenemos el municipio de *San Pedro Del Pinatar*, que tampoco tiene correspondencia con la parte de Idealista. No se hace un cambio para hacerlos coincidir porque *San Pedro Del Pinatar* sí que es un municipio, como es de esperar al aparecer en el CREM, sin embargo,  *La Manga Del Mar Menor* es una zona geográfica que pertenece a dos municipios, a *San Javier* por el norte y a *Cartagena* por el sur. Lo mejor para continuar es quitarlos del estudio posterior (lo hacemos con un INNER JOIN).

In [3]:
#Cambiamos los guiones por barras bajas en el municipio para que tenga el mismo formato en ambos dataframe
df_idealista=(df_idealista
                #Cambiamos los guiones por barras bajas en el municipio para que coincidan en ambas partes
                .withColumn("municipio", F.regexp_replace(F.col("municipio"), "-", "_"))
             )

df_join=(df_idealista
            .join(df_crem
                    ,on=["municipio"]
                    ,how="inner"
                 )
            )

Omitiremos la parte de pintar el esquema, pero, sabemos que todos los campos de estadística son doubles y los que vienen de Idealista son int salvo *municipio* y *estado_ocupacion* que son categóricos, y el campo *price* que es **string**. Primero cambiamos el campo precio de tipo, y luego aplicamos una función de One-Hot Coding para pasar esas variables categóricas a tantas variables como categorías hay, caliendo 1 en la columna correspondiente de la categoría, estas variables enteras serán enteras y no bit o booleanas porque es un formato más adecuado y compatible con librerías o modelos de ML.

In [4]:
#Creamos una función para hacer One-Hot Coding sobre un dataframe y un campo
def one_hot_coding(df, col_name):
    #Sacamos las diferentes categorias
    categorias = [ row[col_name] for row in df.select(col_name).distinct().collect()]

    #Creamos una columna por categoría
    for cat in categorias:
        #Usamos como prefijo la columna original y como sifijo el nombre de la categoria
        df = df.withColumn(f"{col_name}_{cat}", F.when(F.col(col_name)==cat, F.lit(1))
                                                    .otherwise(F.lit(0))
                        )

    #Borramos la columna original
    
    return df.drop(col_name)

In [5]:
#Aplicamos la funcion sobre estado_ocupacion
df_join=one_hot_coding(df_join, "estado_ocupacion")

#Aplicamos la funcion sobre municipio
df_join=one_hot_coding(df_join, "municipio")

In [6]:
#Damos un formato adecuado a ciertos campos
df_join=(df_join
            .withColumn("price", F.col("price").cast("double"))
            .withColumn("metros_cuadrados_construidos", F.col("metros_cuadrados_construidos").cast("double"))
            .withColumn("habitaciones", F.col("habitaciones").cast("double"))
            .withColumn("numero_banos", F.col("numero_banos").cast("double"))
        )

Una vez tenemos todos los datos filtrados, los campos formateados de forma que sean adecuados para aplicar inferencia, o entrenar modelos de ML guardamos el dataset para aplicar ciencia de datos en un trabajo posterior.

In [7]:
# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos, ordenadas)
df_normalized = tfm_utils.normalize_df(df_join)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_normalized)

,2015_distribucion_de_la_renta_p80_p20,2015_edad_media_poblacion,2015_indice_de_gini,2015_porcentaje_de_hogares_unipersonales,2015_porcentaje_de_poblacion_de_65_o_mas,2015_porcentaje_de_poblacion_de_menos_de_18,2015_porcentaje_poblacion_espanola,2015_renta_bruta_media_por_persona,2015_renta_neta_media_por_persona,2015_tamano_medio_del_hogar,...,price,tiene_aire_acondicionado,tiene_ascensor,tiene_balcon,tiene_calefaccion,tiene_garaje,tiene_jardin,tiene_piscina,tiene_terraza,tiene_trastero
0,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,550000.0,1,0,1,1,1,0,0,1,1
1,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,175000.0,1,0,0,0,1,0,0,0,0
2,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,250000.0,0,1,0,0,1,0,0,0,0
3,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,1250000.0,1,0,0,1,1,1,1,0,0
4,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,90000.0,0,0,0,0,0,0,0,1,1
5,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,135000.0,0,0,0,0,1,0,0,0,1
6,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,660000.0,1,0,0,0,1,1,1,1,0
7,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,375000.0,1,0,0,0,1,1,1,1,0
8,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,195000.0,0,0,0,0,0,0,0,0,0
9,3.4,39.5,36.2,23.7,15.1,20.5,89.0,12.013,10.066,2.8,...,235000.0,1,0,0,1,0,1,1,1,0


In [8]:
# Ruta destino de la Delta Table en la capa Silver
ml_table_name = "ml.idealista_crem.dataset_to_apply_ml"
delta_path = tfm_utils.full_table_path(ml_table_name)

print(f"Escribiendo {df_normalized.count()} registros en la Delta Table: {ml_table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_normalized
    .write
    .option("overwriteSchema","true")
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 10509 registros en la Delta Table: ml.idealista_crem.dataset_to_apply_ml
Destino: /tfm/delta_lake/ml/idealista_crem/dataset_to_apply_ml
¡Escritura en Delta Lake completada con éxito!
